# 02 Gateway + Agent API Foundations (OpenClaw, 2026)

## What This Lesson Is
Use OpenClaw as a Python-callable agent gateway via the OpenAI Chat Completions protocol.

## Scientific Lens
- Concept: Pythonic gateway integration with explicit agent/session headers.
- Measure: API call success rate with stable session continuity.
- Validity Limit: Endpoint availability does not imply downstream provider correctness.


## How It Works
1. Construct OpenAI-compatible client against OpenClaw /v1 endpoint.
2. Bind agent/session identity via headers and user field.
3. Compare two calls with same user to validate session continuity semantics.


In [ ]:
import os
from openai import OpenAI

OPENCLAW_BASE_URL = os.getenv("OPENCLAW_BASE_URL", "http://127.0.0.1:18789").rstrip("/")
OPENCLAW_TOKEN = os.getenv("OPENCLAW_GATEWAY_TOKEN") or os.getenv("OPENAI_API_KEY") or ""
OPENCLAW_TOKEN_SOURCE = (
    "OPENCLAW_GATEWAY_TOKEN" if os.getenv("OPENCLAW_GATEWAY_TOKEN")
    else ("OPENAI_API_KEY" if os.getenv("OPENAI_API_KEY") else "<missing>")
)
OPENCLAW_AGENT_ID = os.getenv("OPENCLAW_AGENT_ID", "main")

print("OPENCLAW_BASE_URL:", OPENCLAW_BASE_URL)
print("OPENCLAW_TOKEN source:", OPENCLAW_TOKEN_SOURCE)
print("OPENCLAW_AGENT_ID:", OPENCLAW_AGENT_ID)


def build_client() -> OpenAI:
    # OpenClaw exposes an OpenAI-compatible Chat Completions endpoint at /v1/chat/completions.
    # Docs: https://docs.openclaw.ai/gateway/openai-http-api
    return OpenAI(base_url=f"{OPENCLAW_BASE_URL}/v1", api_key=OPENCLAW_TOKEN or "local-dev-token")


def ask_openclaw(prompt: str, user: str = "lesson-user", temperature: float = 0.2) -> str:
    client = build_client()
    resp = client.chat.completions.create(
        model="openclaw",
        messages=[{"role": "user", "content": prompt}],
        user=user,
        temperature=temperature,
        extra_headers={"x-openclaw-agent-id": OPENCLAW_AGENT_ID},
    )
    return resp.choices[0].message.content or ""


## Code Walkthrough
- `Deterministic Demo` defines and validates the decision logic.
- `Live Demo` executes a real OpenClaw agent call through the OpenAI-compatible gateway API.


In [ ]:
# Deterministic Demo
session_keys = ["team-a:user-17", "team-a:user-17", "team-a:user-42"]
stable = session_keys[0] == session_keys[1] and session_keys[0] != session_keys[2]
print("session_stability_contract:", stable)
assert stable


In [ ]:
# Live Demo
try:
    answer = ask_openclaw(
        "In 3 bullets, explain the role of x-openclaw-agent-id and user in session routing.",
        user="team-a:user-17",
    )
    print(answer)
except Exception as exc:
    print(f"Live demo call failed: {exc}")
    print("Set OPENCLAW_GATEWAY_TOKEN in .env (or export OPENAI_API_KEY) and rerun.")


## Applied Labs
1. Add support for explicit `x-openclaw-session-key` and test session pinning behavior.
2. Run two different agent IDs (`main`, `beta`) and compare style/behavior.
3. Measure latency over 20 calls and chart p50/p95.

## Validation Checklist
- Client uses OpenClaw gateway API, not shell process control.
- Session continuity assumptions are explicit and tested.
- Live demo skips cleanly when token is missing.

## Further Reading
- OpenClaw OpenAI API: https://docs.openclaw.ai/gateway/openai-http-api
- OpenClaw start docs: https://docs.openclaw.ai/start/openclaw
